# Additional optimizer confirmation experiments

This notebook runs only the follow-up experiments identified in the reviewed README. It writes **`results_additional/`**, never overwrites the verified `results/` directory, and creates a zip for download. A CUDA runtime is required; a GPU with more than 16 GiB is preferred for width 4,096.

## 1. Checkout and provenance

Push the latest `colab-results` branch before running. Then select **Runtime → Change runtime type → GPU** and run all cells.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/LokeshJatangi/transformer_optimizer_benchmark.git"
BRANCH = "colab-results"
WORKDIR = Path("/content/transformer-optimizer-benchmark")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(WORKDIR)],
    check=True,
)
os.chdir(WORKDIR)
source_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
os.environ["SOURCE_COMMIT"] = source_commit
print("Checked out source commit:", source_commit)

## 2. Environment, tests, and memory preflight

In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["python", "-m", "pytest", "-q"], check=True)

import torch
from additional_experiments import estimated_parameter_count, training_memory_estimate

assert torch.cuda.is_available(), "Enable a GPU runtime"
device_name = torch.cuda.get_device_name(0)
total_bytes = torch.cuda.get_device_properties(0).total_memory
parameter_count = estimated_parameter_count(65, 4096, 128)
persistent_bytes = training_memory_estimate(parameter_count)
print("GPU:", device_name)
print(f"GPU memory: {total_bytes / 2**30:.2f} GiB")
print(f"Width-4096 parameters: {parameter_count:,}")
print(f"Estimated persistent fp32 AdamW state: {persistent_bytes / 2**30:.2f} GiB")
print("The runner uses micro-batch 1, accumulation 8, activation checkpointing, and foreach=False at width 4096.")

## 3. Run the confirmation suite

The scheduler grid is fully replicated over three initialization/data-order seeds. Validation uses 32 batches per seed. Widths 2,048 and 4,096 each receive five learning rates and three seeds. Progress logs are rewritten after every run. If width 4,096 cannot fit, that limitation is recorded explicitly while all completed evidence remains downloadable.

In [ ]:
from additional_experiments import run_additional

metrics = run_additional()
print("Completed in", metrics["timing"]["total_readable"])
print("Three-seed scheduler winner:", metrics["scheduler_confirmation"]["winner"])
for width, info in metrics["width_confirmation"].items():
    print("Width", width, "status:", info["status"],
          "best LR:" if info["status"] == "complete" else "reason:",
          info["best"]["lr"] if info["status"] == "complete" else info["reason"])

## 4. Audit the bundle

In [ ]:
import csv
import json

from additional_experiments import ADDITIONAL_PLOTS, validate_additional_metrics

results_dir = Path("results_additional")
saved = json.loads((results_dir / "metrics.json").read_text())
validate_additional_metrics(saved)
required = ["metrics.json", "run_log.json", "run_log.csv", "SUMMARY.md",
            "retained_model.pt", *ADDITIONAL_PLOTS]
missing = [name for name in required if not (results_dir / name).exists()]
assert not missing, missing
run_log = json.loads((results_dir / "run_log.json").read_text())
with (results_dir / "run_log.csv").open(newline="") as handle:
    csv_rows = list(csv.DictReader(handle))
assert len(csv_rows) == len(run_log)
unexpected_failures = [row for row in run_log if row["status"] != "ok"
                       and "additional_width_confirmation" not in row["experiment"]]
assert not unexpected_failures, unexpected_failures
print((results_dir / "SUMMARY.md").read_text())
print(f"Validated {len(run_log)} logged runs and {len(required)} required artifacts.")

## 5. Download `results_additional/`

The zip contains the folder itself. Extract it at the repository root and keep the directory name unchanged.

In [ ]:
from google.colab import files

archive = shutil.make_archive(
    "/content/results_additional", "zip",
    root_dir=WORKDIR, base_dir="results_additional",
)
print("Created", archive)
files.download(archive)